<a href="https://colab.research.google.com/github/Eng-Alaa-Mohamed-Ali/Eng-Alaa-Mohamed-Ali/blob/main/Alaa_Mohamed_Ali_telepot_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd && curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,497 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,986 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-bac

In [ ]:
!pip install colab-xterm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.6/115.6 kB 12.7 MB/s eta 0:00:00


In [1]:
import colabxterm

%reload_ext colabxterm
%xterm


#ollama serve
#ollama pull Eomer/gpt-3.5-turbo


ModuleNotFoundError: No module named 'colabxterm'

In [ ]:
!pip install -U langchain-ollama

In [ ]:
from langchain_ollama import ChatOllama

llm_model=ChatOllama(
    model="granite3.3:2b",
    temperature=0.0,
    streaming=True,

)

In [ ]:
llm_model.invoke("Hello, tell me 3 names of Arab countries.").content

'\nOf course! Here are three names of Arab countries:\n\n1. Egypt\n2. Saudi Arabia\n3. Iraq'

In [ ]:
!pip install telebot


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 27.8 MB/s eta 0:00:00


In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 19.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp "/content/drive/MyDrive/Colab Notebooks/Practical Malware Analysis.pdf" "/content/BOOK.pdf"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import re
import unicodedata

def clean_text_for_embedding(text):
    """
    Comprehensive text cleaning function for embeddings.
    Handles Unicode issues, formatting problems, and ensures clean text for processing.

    Args:
        text (str): Raw text to clean

    Returns:
        str: Cleaned text safe for embeddings and JSON serialization
    """

    # Handle None or non-string inputs
    if not text or not isinstance(text, str):
        return ""

    # Step 1: Handle bytes if accidentally passed
    if isinstance(text, bytes):
        text = text.decode('utf-8', errors='ignore')

    # Step 2: Remove problematic Unicode surrogate characters
    # This fixes the '\ud835' error specifically
    text = re.sub(r'[\ud800-\udfff]', ' ', text)

    # Step 3: Remove other problematic Unicode ranges
    text = re.sub(r'[\u0000-\u0008\u000b\u000c\u000e-\u001f\u007f-\u009f]', '', text)  # Control characters
    text = re.sub(r'[\ufeff\ufffe\uffff]', '', text)  # BOM and other problematic chars

    # Step 4: Handle mathematical and special symbols
    text = re.sub(r'[\U0001D400-\U0001D7FF]', '[MATH]', text)  # Mathematical symbols
    text = re.sub(r'[\u2190-\u21FF]', '[ARROW]', text)  # Arrows
    text = re.sub(r'[\u2200-\u22FF]', '[SYMBOL]', text)  # Mathematical operators

    # Step 5: Normalize Unicode characters
    try:
        text = unicodedata.normalize('NFKC', text)
    except Exception:
        # If normalization fails, try a more aggressive approach
        text = ''.join(char for char in text if ord(char) < 65536)  # Keep only BMP characters

    # Step 6: Clean up whitespace and formatting
    text = re.sub(r'\s+', ' ', text)  # Multiple whitespace to single space
    text = re.sub(r'\n+', ' ', text)  # Multiple newlines to single space
    text = re.sub(r'\t+', ' ', text)  # Tabs to spaces

    # Step 7: Remove excessive punctuation
    text = re.sub(r'[.]{3,}', '...', text)  # Multiple dots to ellipsis
    text = re.sub(r'[-]{2,}', '--', text)   # Multiple dashes to double dash
    text = re.sub(r'[!]{2,}', '!', text)    # Multiple exclamations to single
    text = re.sub(r'[?]{2,}', '?', text)    # Multiple questions to single

    # Step 8: Final safety check - encode/decode to catch any remaining issues
    try:
        text = text.encode('utf-8', errors='ignore').decode('utf-8', errors='ignore')
    except Exception:
        # Ultimate fallback - keep only ASCII characters
        text = ''.join(char for char in text if ord(char) < 128)

    # Step 9: Final cleanup
    text = text.strip()

    # Step 10: Test JSON serialization (what was causing the original error)
    try:
        import json
        json.dumps(text)  # This will raise an error if text still has issues
    except Exception:
        # If JSON serialization still fails, use ASCII-only fallback
        text = ''.join(char for char in text if ord(char) < 128)
        text = re.sub(r'\s+', ' ', text).strip()

    return text


# Example usage:
def clean_document_list(documents):
    """
    Clean a list of LangChain documents

    Args:
        documents: List of Document objects with page_content attribute

    Returns:
        List of cleaned text strings
    """
    cleaned_texts = []

    for doc in documents:
        # Extract text content
        if hasattr(doc, 'page_content'):
            raw_text = doc.page_content
        else:
            raw_text = str(doc)

        # Clean the text
        clean_text = clean_text_for_embedding(raw_text)

        # Only keep non-empty texts with sufficient content
        if clean_text and len(clean_text) > 10:
            cleaned_texts.append(clean_text)

    return cleaned_texts


# Test function to verify cleaning works
def test_cleaning():
    """Test the cleaning function with problematic characters"""

    # Test cases with problematic Unicode
    test_texts = [
        "Normal text should pass through fine.",
        "Text with surrogates: \ud835\udc00\ud835\udc01 should be cleaned",
        "Math symbols: 𝓐𝓑𝓒 should be replaced",
        "Multiple   spaces    should    be    cleaned",
        "Control\x00characters\x01should\x02be\x03removed",
        None,  # None input
        "",    # Empty string
        123,   # Non-string input
    ]

    print("Testing text cleaning function:")
    for i, text in enumerate(test_texts):
        try:
            cleaned = clean_text_for_embedding(text)
            print(f"Test {i+1}: SUCCESS - '{cleaned}'")

            # Verify JSON serialization works
            import json
            json.dumps(cleaned)
            print(f"  JSON serialization: OK")

        except Exception as e:
            print(f"Test {i+1}: FAILED - {e}")

    return "All tests completed"

In [ ]:
!pip install -q langchain langchain-community faiss-cpu sentence-transformers accelerate
from langchain_community.document_loaders import PyPDFLoader
#Load documents
loader = PyPDFLoader("/content/BOOK.pdf")
document_list = loader.load()

Book_content = []

for doc in document_list:
    # Extract text content
    raw_text = doc.page_content if hasattr(doc, 'page_content') else str(doc)

    # Clean the text using our single function
    clean_text = clean_text_for_embedding(raw_text)

    # Only keep texts with sufficient content
    if clean_text and len(clean_text) > 20:
        Book_content.append(clean_text)

Book_content = '\n\n'.join(Book_content)

In [ ]:
Book_content[:800]

"PRAISE FOR PRACTICAL MALWARE ANALYSIS “An excellent crash course in malware analysis.” —Dino Dai Zovi, INDEPENDENT SECURITY CONSULTANT “. . . the most comprehensive guide to analysis of malware, offering detailed coverage of all the essential skills re quired to understand the specific challenges presented by modern malware.” —Chris Eagle, SENIOR LECTURER OF COMPUTER SCIENCE, NAVAL POSTGRADUATE SCHOOL “A hands-on introduction to malware analysis. I'd recommend it to anyone who wants to dissect Windows malware.” —Ilfak Guilfanov, C REATOR OF IDA PRO “. . . a great introduction to malware analysis. All chapters contain detailed technical explanations and hands-on lab exercises to get you immediate exposure to real malware.” —Sebastian Porst, G OOGLE SOFTWARE ENGINEER “. . . brings reverse-en"

In [ ]:
!pip install -q langchain langchain-community faiss-cpu sentence-transformers accelerate

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

#Split into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_text(Book_content)

#embedder
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

#vector database
vectorstore = FAISS.from_texts(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

/tmp/ipykernel_4625/4217496515.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def pre_retrieval_rag(q):
  docs = retriever.invoke(q)
  context = "\n".join([d.page_content for d in docs])
  return context

In [ ]:
import telebot
from langchain_ollama import ChatOllama

API_TOKEN = "8549364691:AAFcNm_cf5XMRj5rOzwIjyUP7YXuaRt2M5Q"
bot = telebot.TeleBot(API_TOKEN)

# Pull the Ollama model if not already available
!ollama pull granite3.3:2b

# Re-initialize llm_model after ensuring the model is pulled
llm_model=ChatOllama(
    model="granite3.3:2b",
    temperature=0.0,
    streaming=True,
)

@bot.message_handler(func=lambda message: True)
def echo_all(message):
    user_text = message.text
    chat_id = message.chat.id
    context = pre_retrieval_rag(user_text)
    prompt = f"""
    Answer only from the provided Context.
    If you don't know the answer, just say that you don't know, don't try to make up an answer.
    If there is a context answer based on it if not Answer 'Not provided context'
    Context: {context}
    Q: {user_text}
    """
    response = llm_model.invoke(prompt).content
    bot.send_message(chat_id, response)

bot.polling()